# M3 — Priority (review check)

Exact priority is unreliable (labels are inconsistent, so the ceiling is ~50%). Rather than assert a priority, this model estimates one and is used to **flag** cases where the agent's priority is `FLAG_GAP` (=2) or more levels away. Inbound features only — no `sla_target_hours` (which is derived from priority) and no plan proxy.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
df       = P.load_clean()
FEATURES = ["case_summary", "case_type", "channel"]   # no sla_target (derived from priority), no plan
TARGET   = "priority"
classes  = P.PRIORITIES
FLAG_GAP = 2                                           # flag when estimate and agent differ this much

train, _, test = P.time_split(df)
model = P.make_clf([P.t_word(), P.t_cat(["case_type", "channel"])]).fit(train[FEATURES], train[TARGET])
pred  = model.predict(test[FEATURES])

baseline = df[TARGET].value_counts(normalize=True).iloc[0]
metrics  = P.evaluate(test[TARGET], pred, classes, baseline=baseline)
metrics.update(P.ordinal_extras(test[TARGET], pred, classes))

# How often would the >=2-level rule fire against the recorded priority?
rank = P.PRIORITY_RANK
gap  = np.abs(test[TARGET].map(rank).to_numpy() - np.array([rank[p] for p in pred]))
metrics["flag_rate"] = round(float((gap >= FLAG_GAP).mean()), 3)
metrics

In [ ]:
proba = model.predict_proba(test[FEATURES])
heldout = test[["case_id", "case_summary", "case_type", "channel", TARGET]].copy().reset_index(drop=True)
heldout["predicted"]  = pred
heldout["confidence"] = proba.max(axis=1).round(3)

deployed = P.make_clf([P.t_word(), P.t_cat(["case_type", "channel"])]).fit(df[FEATURES], df[TARGET])
P.save_bundle("m3_priority_review", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "multiclass",
    "label": "Priority — review check", "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "D", "approach": "priority double-check (ordinal, flag mismatches)",
             "split": "time", "flag_gap": FLAG_GAP, "priorities": classes,
             "inputs": "summary + case_type + channel (no sla_target, no plan proxy)"}})